# 00 — Setup Google Colab

**Projeto:** Predição de vitória de round em CS:GO (ML Clássico)

Este notebook deve rodar **do zero** no Colab (issue #3).

Documentação: `Docs/guias/03-google-colab-como-usar.md`

## 1. (Opcional) Clonar repositório

In [ ]:
# Descomente se abrir o Colab vazio e quiser clonar o repo:
# !git clone https://github.com/Theordep/projeto-machine-learning-csgo.git
# %cd projeto-machine-learning-csgo

## 2. Instalar dependências

In [ ]:
!pip install -q -r requirements-colab.txt

## 3. Autenticação Kaggle (se necessário)

Faça upload do `kaggle.json` ou use Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`).

In [ ]:
from pathlib import Path
import json
import os

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

if not (kaggle_dir / "kaggle.json").exists():
    try:
        from google.colab import userdata  # type: ignore
        creds = {
            "username": userdata.get("KAGGLE_USERNAME"),
            "key": userdata.get("KAGGLE_KEY"),
        }
        (kaggle_dir / "kaggle.json").write_text(json.dumps(creds))
        print("kaggle.json criado a partir de Colab Secrets.")
    except Exception:
        print("Upload manual: Files → kaggle.json → copiar para ~/.kaggle/")
else:
    print("kaggle.json já existe.")

os.chmod(kaggle_dir / "kaggle.json", 0o600) if (kaggle_dir / "kaggle.json").exists() else None

## 4. Download do dataset (kagglehub)

In [ ]:
import shutil
import kagglehub
import pandas as pd

DATASET = "christianlillelund/csgo-round-winner-classification"
DATA_RAW = Path("/content/data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

cache_path = Path(kagglehub.dataset_download(DATASET))
print("Cache kagglehub:", cache_path)

csv_path = None
for src in cache_path.rglob("*.csv"):
    dest = DATA_RAW / src.name
    shutil.copy2(src, dest)
    print("Copiado →", dest)
    if src.name == "csgo_round_snapshots.csv" or csv_path is None:
        csv_path = dest

if csv_path is None:
    raise FileNotFoundError(f"Nenhum CSV em {cache_path}")

CSV_PATH = csv_path
print("Arquivo principal:", CSV_PATH)

## 5. Carregar e inspecionar

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("Nulos:", df.isna().sum().sum())
print("\nColunas (primeiras 15):", list(df.columns[:15]))
df.head()

In [ ]:
print("Vencedor do round:")
print(df["round_winner"].value_counts())
print("\nMapas:")
print(df["map"].value_counts().head())

---
Próximo passo: **`01_EDA.ipynb`** (issue #4)